In [ ]:
import os
import requests
import time
import math
from dataclasses import dataclass
import pickle
import joblib

import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import pandas as pd
import re
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"正在使用设备: {DEVICE}")

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float
    bias: bool = True

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"序列长度 {t} 不能超过设定的 block_size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx

generator_chars = []
generator_stoi = {}
generator_itos = {}
generator_vocab_size = 0

def load_generator_tokenizer_data(file_path):
    global generator_chars, generator_stoi, generator_itos, generator_vocab_size
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.read()
    generator_chars = sorted(list(set(data)))
    generator_vocab_size = len(generator_chars)
    generator_stoi = {ch: i for i, ch in enumerate(generator_chars)}
    generator_itos = {i: ch for i, ch in enumerate(generator_chars)}
    print(f"generator vocabulary size: {generator_vocab_size}")

def encode_generator(s):
    return [generator_stoi[c] for c in s if c in generator_stoi]

def decode_generator(l):
    return ''.join([generator_itos[i] for i in l if i in generator_itos])

class FakeNewsLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers):
        super(FakeNewsLSTM, self).__init__()
        self.text_embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=n_layers, dropout=0.2 if n_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, text_seq):
        x = self.text_embedding(text_seq)
        _, (h_n, _) = self.lstm(x)
        lstm_output = h_n[-1]
        return self.fc(lstm_output)

def tokenize_classifier(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\\s]', '', text)
    return text.split()

def encode_tokens_classifier(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

if __name__ == "__main__":
    generator_training_data_path = os.path.join("model", "GPTmodel_Newdataset", "Fake_news", "fake_news_formatted.txt")
    generator_model_path = os.path.join("model", "GPTmodel_Newdataset", "Fake_news", "GPTmodel_trained_Newdataset.pth")

    classifier_model_path = os.path.join("code", "best_pure_classification_model.pkl")
    classifier_training_data_path = os.path.join("data", "new_class_dataset.csv")


    print("\nLoading Generator Model")
    BLOCK_SIZE_GEN = 256
    N_EMBD_GEN = 384
    N_HEAD_GEN = 6
    N_LAYER_GEN = 6
    DROPOUT_GEN = 0.2

    load_generator_tokenizer_data(generator_training_data_path)

    gpt_config = GPTConfig(
        block_size=BLOCK_SIZE_GEN, vocab_size=generator_vocab_size,
        n_layer=N_LAYER_GEN, n_head=N_HEAD_GEN, n_embd=N_EMBD_GEN, dropout=DROPOUT_GEN
    )
    model_generator = GPT(gpt_config)
    model_generator.load_state_dict(torch.load(generator_model_path, map_location=DEVICE))
    model_generator.to(DEVICE)
    model_generator.eval()
    print(f"Generator Model successfully loaded from{generator_model_path} ")

    
    class_data_for_vocab = pd.read_csv(classifier_training_data_path)
    class_data_for_vocab['input'] = class_data_for_vocab['input'].astype(str).str.slice(0, 300)
    class_data_for_vocab['tokens'] = class_data_for_vocab['input'].apply(tokenize_classifier)
    all_tokens_for_vocab = [token for tokens_list in class_data_for_vocab['tokens'] for token in tokens_list]
    token_counts_for_vocab = Counter(all_tokens_for_vocab)
    n_words_classifier = 10000
    vocab_classifier = {word: i+2 for i, (word, _) in enumerate(token_counts_for_vocab.most_common(n_words_classifier))}
    vocab_classifier['<PAD>'] = 0
    vocab_classifier['<UNK>'] = 1
    print(f"Classifier vocabulary size {len(vocab_classifier)}")


    CLASSIFIER_VOCAB_SIZE = len(vocab_classifier)
    CLASSIFIER_EMBED_DIM = 50
    CLASSIFIER_HIDDEN_DIM = 64
    CLASSIFIER_N_LAYERS = 2

    model_bundle = joblib.load(classifier_model_path)
    model_classifier = model_bundle['model']
    model_classifier.to(DEVICE)
    model_classifier.eval()
    print(f"Classifier Model loaded from {classifier_model_path} ")

    print("\n--- Generating news ---")
    start_string_generator = "[input]trump"
    start_ids = encode_generator(start_string_generator)
    x_input = torch.tensor(start_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    max_tokens_to_generate = 300
    generated_ids = model_generator.generate(
        x_input, max_new_tokens=max_tokens_to_generate, temperature=0.7, top_k=50
    )
    generated_text_full = decode_generator(generated_ids[0].tolist())
    generated_text_article = generated_text_full[len(start_string_generator):]
    print(f"Generator prompt'{start_string_generator}'")
    print(f"Generator response: \n{generated_text_article[:500]}...")

    print("\n--- Classifying the generated news ---")
    classifier_tokens = tokenize_classifier(generated_text_article)
    classifier_encoded_tokens = encode_tokens_classifier(classifier_tokens, vocab_classifier)
    classifier_input_tensor = torch.tensor(classifier_encoded_tokens, dtype=torch.long, device=DEVICE)
    if classifier_input_tensor.ndim == 1:
        classifier_input_tensor = classifier_input_tensor.unsqueeze(0)
    
    if classifier_input_tensor.size(1) == 0:
        print("empty, cant classify")
    else:
        with torch.no_grad():
            prediction_prob = model_classifier(classifier_input_tensor).squeeze()
            predicted_label = (prediction_prob > 0.5).int()
        print(f"Probability output  {prediction_prob.item():.4f}")
        if predicted_label.item() == 1:
            print("Classifier Result: Fake")
        else:
            print("Classifier Result: Real")

正在使用设备: cpu

Loading Generator Model
generator vocabulary size: 275
Generator Model successfully loaded frommodel\GPTmodel_Newdataset\Fake_news\GPTmodel_trained_Newdataset.pth 
Classifier vocabulary size 10002
Classifier Model loaded from code\best_pure_classification_model.pkl 

--- Generating news ---
Generator prompt'trump'
Generator response: 
’s health and his presidential nomination was “expected”. Joel Brown, according to the U.S. state of Congress is also refused to finance minister to be shot on Monday showed by laying a political contract with an early marked on the U.S. Senate negative order to decide with comment. “I think that th...

--- Classifying the generated news ---
Probability output  0.4813
Classifier Result: Real
